In [1]:
import os
from jinja2 import Environment, FileSystemLoader

# Đi thẳng từ thư mục gốc của dự án
project_root = "/home/duoc/workflow/api"
template_dir = os.path.join(project_root, "app/marketing")

In [2]:
import os
import re
import pandas as pd
from jinja2 import Environment, FileSystemLoader

# ==============================================================================
# THIẾT LẬP ĐƯỜNG DẪN HỆ THỐNG
# ==============================================================================
try:
    template_dir = os.path.join(project_root, "app/marketing")
except NameError:
    base_dir = os.getcwd()
    template_dir = os.path.abspath(os.path.join(base_dir, "../../app/marketing")) if "notebooks" in base_dir else os.path.abspath(os.path.join(base_dir, "app/marketing"))

template_path = os.path.join(template_dir, "prompt_templates.j2")

# ==============================================================================
# 1. ĐỌC FILE, DÙNG REGEX SỬA TRIỆT ĐỂ BIẾN LỖI (VÁ LỖI UNDEFINED VOICE)
# ==============================================================================
if os.path.exists(template_path):
    with open(template_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Bước A: Thêm Block Role & System Instruction lên đầu file nếu chưa có
    if "# ROLE & SYSTEM INSTRUCTIONS" not in content:
        system_headers = (
            "# ROLE & SYSTEM INSTRUCTIONS\n"
            "Bạn là một chuyên gia Content Marketing cấp cao kiêm Đại sứ Thương hiệu chuyên nghiệp. "
            "Nhiệm vụ của bạn là tạo ra nội dung dựa trên bộ quy tắc Brand Voice và cấu trúc kịch bản được cung cấp dưới đây.\n\n"
        )
        content = system_headers + content

    # Bước B: Ép kịch bản tuân thủ cấu hình động (Hỗ trợ quét lại nếu file đã lỗi sẵn)
    content = content.replace("- Mỗi đoạn 3–5 câu", "- Mỗi đoạn tối đa {{ brand_voice.formatRules.paragraphMaxSentences | default(3, true) }} câu")
    content = content.replace(
        '5. CTA mềm — "Tìm hiểu thêm", "Đọc tiếp", "Chia sẻ suy nghĩ của bạn"', 
        "5. CTA — Viết theo phong cách {{ brand_voice.ctaStyle.style | default('direct', true) }} của thương hiệu"
    )

    # Bước C: VÁ LỖI BIẾN: Quét toàn bộ file và chuyển toàn bộ {{ voice.xxx }} sang {{ brand_voice.xxx }}
    # Bất kể file đang ở trạng thái lỗi nào, Regex này sẽ dọn sạch biến voice mồ côi
    content = re.sub(r'{{\s*voice\.', '{{ brand_voice.', content)

    # Bước D: Chuyển đổi các tiêu đề cũ sang cấu trúc Phân vùng (Section) hệ thống
    content = content.replace("# BRAND VOICE", "# SECTION 1: BRAND VOICE RULES\nLuôn luôn tuân thủ nghiêm ngặt các quy tắc về giọng điệu và phong cách viết này trong suốt bài viết:")
    content = content.replace("## Examples — Ví dụ mẫu", "# SECTION 2: FEW-SHOT EXAMPLES (VÍ DỤ MẪU CHUẨN)")
    content = content.replace("## BLOG POST TEMPLATE", "# SECTION 3: CONTENT PRODUCTION TEMPLATE (BLOG POST)")
    content = content.replace("## COLD EMAIL TEMPLATE", "# SECTION 3: CONTENT PRODUCTION TEMPLATE (COLD EMAIL)")
    content = content.replace("## INSTAGRAM CAPTION TEMPLATE", "# SECTION 3: CONTENT PRODUCTION TEMPLATE (INSTAGRAM CAPTION)")

    # Bước E: Tái cấu trúc khối USER INPUT dính chữ sang XML chuẩn và xử lý khoảng trắng Jinja2
    xml_user_input_pattern = r"### USER INPUT.*?(?=\n---|\nViết|\Z)"
    
    def transform_to_xml(match):
        block = match.group(0)
        # Khử kỹ thuật tước khoảng trắng thụ động '{\%-' của Jinja2 vốn là nguyên nhân gây dính chữ dữ liệu
        block_clean_whitespace = block.replace("{%- if user_input.keywords %}", "{% if user_input.keywords %}")\
                                      .replace("{%- if user_input.audience %}", "{% if user_input.audience %}")\
                                      .replace("{%- if user_input.productName %}", "{% if user_input.productName %}")\
                                      .replace("{%- if user_input.toneHint %}", "{% if user_input.toneHint %}")\
                                      .replace("{%- if user_input.recipientName %}", "{% if user_input.recipientName %}")\
                                      .replace("{%- if user_input.painPoint %}", "{% if user_input.painPoint %}")\
                                      .replace("{%- if user_input.ctaAction %}", "{% if user_input.ctaAction %}")\
                                      .replace("{%- if user_input.visualDescription %}", "{% if user_input.visualDescription %}")\
                                      .replace("{%- if user_input.ctaType %}", "{% if user_input.ctaType %}")\
                                      .replace("{%- endif %}", "{% endif %}")
        
        # Bọc thẻ XML đóng/mở cùng mệnh lệnh hành động thực thi tối ưu cho mô hình nhỏ
        cleaned_block = block_clean_whitespace.replace("### USER INPUT", "# SECTION 4: DATA INPUT & EXECUTION\nDữ liệu đầu vào từ người dùng được cung cấp trong thẻ XML dưới đây:\n\n<user_input>")
        cleaned_block += "\n</user_input>\n\n👉 **MỆNH LỆNH THỰC THI:** Dựa vào toàn bộ hướng dẫn hệ thống và dữ liệu trong thẻ <user_input>, hãy thực hiện yêu cầu ngay bây giờ. Output chỉ trả về nội dung hoàn chỉnh, không kèm lời thoại của AI."
        return cleaned_block

    content = re.sub(xml_user_input_pattern, transform_to_xml, content, flags=re.DOTALL)
    
    # Bước F: Dọn dẹp hoàn toàn các chỉ thị kết thúc dòng thô thừa thãi
    content = content.replace("---\nViết bài blog ngay bây giờ.", "")
    content = content.replace("---\nViết cold email ngay bây giờ.", "")
    content = content.replace("---\nViết Instagram caption ngay bây giờ.", "")

    # Bước G: Đẩy dữ liệu sạch đã sửa đổi triệt để xuống ổ cứng
    with open(template_path, "w", encoding="utf-8") as f:
        f.write(content.strip())
    print(f"🔹 Đã sửa toàn bộ lỗi biến 'voice' bằng Regex thành công tại: {template_path}")
else:
    print(f"❌ Không tìm thấy file template tại {template_path} để sửa đổi!")

# ==============================================================================
# 2. KHỞI TẠO JINJA2 ENVIRONMENT & LOAD FILE SAU KHI ĐÃ SỬA
# ==============================================================================
env = Environment(
    loader=FileSystemLoader(template_dir),
    trim_blocks=True,
    lstrip_blocks=True
)
template = env.get_template("prompt_templates.j2")

# ==============================================================================
# 3. MOCK DATA 36 THƯƠNG HIỆU (Kiểm thử biên dữ liệu khuyết mảng hệ thống)
# ==============================================================================
mock_brands = {}
for i in range(1, 37):
    b_name = f"Thương_Hiệu_{i}"
    is_bad_data = (i % 7 == 0)
    
    mock_brands[b_name] = {
        "personality": f"Bạn là đại sứ thương hiệu cấp cao của {b_name}." if not is_bad_data else None,
        "tone": { 
            "base": ["confident", "professional"] if not is_bad_data else None, 
            "overrides": {"blog": ["helpful"], "email": ["respectful"], "social": ["energetic"]} if not is_bad_data else None 
        },
        "style": { "sentenceLength": "short" if not is_bad_data else None, "voice": "active", "perspective": "second" },
        "vocabulary": { 
            "wordsToUse": ["giải pháp", "đột phá"] if i % 2 == 0 else None, 
            "wordsToAvoid": ["tệ", "kém chất lượng", "lừa đảo"] if not is_bad_data else None 
        },
        "formatRules": { "paragraphMaxSentences": 3 if not is_bad_data else None, "useEmoji": True, "useHashtags": True if i % 3 == 0 else False },
        "ctaStyle": { "style": "direct" if not is_bad_data else None, "phrases": ["Khám phá ngay"] if not is_bad_data else None },
        "examples": None if is_bad_data else []
    }

test_input = {
    "topic": "Sản phẩm chiến lược mới 2026",
    "keywords": ["tối ưu", "bền bỉ"],
    "audience": "khách hàng mục tiêu",
    "productName": "Alpha Max Pro",
    "recipientName": "Anh Nguyễn Văn A",
    "painPoint": "Tốn quá nhiều thời gian quản lý",
    "ctaAction": "Đặt lịch tư vấn 15 phút"
}

# ==============================================================================
# 4. CHẠY VÒNG LẶP KIỂM THỬ TỰ ĐỘNG VỚI TIÊU CHÍ CHECK THEO CHUẨN MỚI
# ==============================================================================
results = []
content_types = ["blog", "email", "social"]

for b_name, b_voice in mock_brands.items():
    for c_type in content_types:
        status = "Thành công"
        error_details = []
        
        try:
            raw_prompt = template.render(brand_voice=b_voice, content_type=c_type, user_input=test_input)
            clean_prompt = re.sub(r'\n{3,}', '\n\n', raw_prompt).strip()
            
            if "None" in clean_prompt:
                error_details.append("Lỗi dữ liệu trống (chứa chữ 'None')")
            
            if "\n\n\n\n" in clean_prompt:
                error_details.append("Lỗi định dạng (Dư thừa dòng trống liên tiếp)")
                
            # Kiểm thử cấu trúc bọc thẻ XML
            if "<user_input>" not in clean_prompt or "</user_input>" not in clean_prompt:
                error_details.append("Lỗi cấu trúc: Thiếu thẻ XML bọc dữ liệu đầu vào")
                
            # Kiểm thử dính chữ do lỗi nuốt dòng trống thụ động của Jinja2 cũ
            if "Topic:" in clean_prompt and "Keywords:" in clean_prompt:
                if "Topic: " in clean_prompt and "\nKeywords:" not in clean_prompt:
                    error_details.append("Lỗi nén text: Dữ liệu trường đầu vào bị dính chữ, không xuống dòng")
                
            if error_details:
                status = "Cảnh báo lỗi Logic"
                
        except Exception as e:
            status = "Thất bại (Sập Code Kỹ Thuật)"
            error_details.append(str(e))
            
        results.append({
            "Thương Hiệu": b_name,
            "Kênh Content": c_type,
            "Trạng Thái": status,
            "Chi Tiết Lỗi": " | ".join(error_details) if error_details else "Không có lỗi",
            "Độ Dài Prompt (Kí tự)": len(clean_prompt) if status == "Thành công" else 0
        })

# ==============================================================================
# 5. XUẤT DASHBOARD KẾT QUẢ CUỐI CÙNG
# ==============================================================================
df_final = pd.DataFrame(results)

print("="*60)
print("📊 BÁO CÁO THỐNG KÊ CHẤT LƯỢNG PROMPT ENGINE (BẢN VÁ LỖI TOÀN DIỆN)")
print("="*60)
print(df_final["Trạng Thái"].value_counts())
print("-"*60)

df_errors_only = df_final[df_final["Trạng Thái"] != "Thành công"]
if not df_errors_only.empty:
    print(f"⚠️ PHÁT HIỆN {len(df_errors_only)} TRƯỜNG HỢP LỖI CẦN XỬ LÝ:")
    display(df_errors_only.reset_index(drop=True))
else:
    print("🎉 KHÔNG CÒN LỖI! Hệ thống Prompt Engine đạt chuẩn Enterprise sạch 100%. Ready cho Production chạy dự án thực tế!")

🔹 Đã sửa toàn bộ lỗi biến 'voice' bằng Regex thành công tại: /home/duoc/workflow/api/app/marketing/prompt_templates.j2
📊 BÁO CÁO THỐNG KÊ CHẤT LƯỢNG PROMPT ENGINE (BẢN VÁ LỖI TOÀN DIỆN)
Trạng Thái
Thành công    108
Name: count, dtype: int64
------------------------------------------------------------
🎉 KHÔNG CÒN LỖI! Hệ thống Prompt Engine đạt chuẩn Enterprise sạch 100%. Ready cho Production chạy dự án thực tế!


In [3]:
import os
import re
from jinja2 import Environment, FileSystemLoader

# ==============================================================================
# 1. LOAD TEMPLATE VỚI CẤU HÌNH KHỬ WHITESPACE TỰ ĐỘNG
# ==============================================================================
template_dir = os.path.join(project_root, "app/marketing")

env = Environment(
    loader=FileSystemLoader(template_dir),
    trim_blocks=True,      # Tự động xóa dòng trống sau các block lệnh {% ... %}
    lstrip_blocks=True     # Tự động xóa khoảng trắng/tab ở đầu dòng trước block lệnh
)
template = env.get_template("prompt_templates.j2")

# ==============================================================================
# 2. DATA INPUT (Giữ nguyên cấu trúc của bạn)
# ==============================================================================
brand_voice = {
    "personality": "Bạn là chuyên gia marketing...",
    "tone": { "base": ["confident"], "overrides": {"blog": ["helpful"]} },
    "style": { "sentenceLength": "short", "voice": "active", "perspective": "second" },
    "vocabulary": { "wordsToUse": [], "wordsToAvoid": [] },
    "formatRules": { "paragraphMaxSentences": 3, "useEmoji": True },
    "ctaStyle": { "style": "direct", "phrases": [] },
    "examples": []
}

# ==============================================================================
# 3. RENDER VÀ ÁP DỤNG CHỐT CHẶN REGEX PYTHON
# ==============================================================================
# Bước A: Render nạp data vào khuôn .j2 từ ổ cứng
raw_prompt = template.render(
    brand_voice=brand_voice,
    content_type="blog",           # "blog" | "email" | "social"
    user_input={
        "topic": "Giày chạy bộ mới",
        "keywords": ["nhẹ", "bền"],
        "audience": "runner beginner",
        "productName": "Air Zoom Pegasus"
    }
)

# Bước B: Chốt chặn Regex - Thay thế cụm từ 3 dấu xuống dòng trở lên thành đúng 2 dấu
clean_prompt = re.sub(r'\n{3,}', '\n\n', raw_prompt).strip()

# ==============================================================================
# 4. IN KẾT QUẢ ĐỂ CHECK
# ==============================================================================
print(clean_prompt)

# ROLE & SYSTEM INSTRUCTIONS
Bạn là một chuyên gia Content Marketing cấp cao kiêm Đại sứ Thương hiệu chuyên nghiệp. Nhiệm vụ của bạn là tạo ra nội dung dựa trên bộ quy tắc Brand Voice và cấu trúc kịch bản được cung cấp dưới đây.

# SECTION 1: BRAND VOICE RULES
Luôn luôn tuân thủ nghiêm ngặt các quy tắc về giọng điệu và phong cách viết này trong suốt bài viết:

## Identity
Bạn là chuyên gia marketing...

## Tone — Giọng điệu cơ bản
Giọng điệu: confident
Luôn duy trì cảm xúc này trong mọi content.

## Style — Cách viết
- Độ dài câu: short
- Giọng văn: active (chủ động/bị động)
- Góc nhìn: second (thứ nhất/nhì/ba)

## Vocabulary Rules — Quy tắc từ vựng

## Format Rules — Quy tắc định dạng
- Đoạn tối đa 3 câu
- Emoji: có
- Hashtag: không

## Call-to-Action — Kêu gọi hành động
Phong cách CTA: direct

## Contextual Tone for blog
Tone điều chỉnh: helpful

# SECTION 3: CONTENT PRODUCTION TEMPLATE (BLOG POST) — Hướng dẫn viết bài blog

### Yêu cầu độ dài
- 800–1200 từ
- Mỗi đoạn tối đa 3 câu

#